# Treinamento inicial dos modelos

## Objetivo
Treinar três modelos no conjunto de treino: `DummyClassifier` como baseline, `LogisticRegression` e `RandomForestClassifier`. O teste permanece separado e não é usado para ajustar ou escolher os modelos nesta etapa.

## Como pensar nos modelos
- **DummyClassifier:** cria uma referência simples; com `most_frequent`, sempre prevê a classe mais comum. Se um modelo sofisticado não superar esse baseline nas métricas adequadas, ele não demonstrou valor.
- **Regressão logística:** apesar do nome, é um classificador. Combina atributos em uma pontuação, converte-a em probabilidade por uma função logística e compara a probabilidade com um limiar (por padrão, 0,5).
- **Random Forest:** combina várias árvores de decisão treinadas com amostras e subconjuntos de atributos. Pode capturar relações não lineares; em contrapartida, tende a ser menos simples de explicar que um modelo linear.

As classes estão desbalanceadas (aproximadamente 11% `yes`), por isso `class_weight="balanced"` será usado inicialmente nos modelos Logistic Regression e Random Forest. Esse peso pode alterar o equilíbrio entre precision e recall; a avaliação decidirá se foi útil.

Os parâmetros são configurações iniciais fixadas antes de olhar o teste. Se fizermos busca ou ajustarmos limiar depois, usaremos validação cruzada estratificada somente dentro do treino. O teste será reservado para a avaliação final.

In [1]:
from pathlib import Path
import sys

import joblib
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from preprocessing import load_dataset, make_preprocessor, prepare_features_target, split_data

MODELS_PATH = PROJECT_ROOT / "results" / "models"
MODELS_PATH.mkdir(parents=True, exist_ok=True)
print(f"Raiz do projeto: {PROJECT_ROOT}")

Raiz do projeto: C:\Users\User\Documents\Codex\2026-11-09\files-pasted-by-the-user-projeto


## Reconstruir a mesma divisão

Usamos as mesmas funções e os mesmos parâmetros do notebook de pré-processamento para garantir que os modelos recebam a mesma divisão reproduzível. O conjunto de teste é separado, mas não será passado a `.fit()`, nem usado para obter previsões nesta etapa.

In [2]:
df = load_dataset()
X, y = prepare_features_target(df)
X_train, X_test, y_train, y_test = split_data(
    X, y, test_size=0.2, random_state=42
)
print(f"Amostras de treino: {len(X_train):,}")
print(f"Amostras de teste reservadas: {len(X_test):,}")
print(f"Rótulos usados no treino: {y_train.value_counts().to_dict()}")

Amostras de treino: 32,950
Amostras de teste reservadas: 8,238
Rótulos usados no treino: {'no': 29238, 'yes': 3712}


## Definir os modelos iniciais

`class_weight="balanced"` atribui mais peso à classe menos frequente durante o ajuste. Isso não garante melhora: pode ajudar a identificar mais `yes`, mas também aumentar falsos positivos. O notebook não escolhe previamente qual modelo será melhor.

O `DummyClassifier` usa o rótulo majoritário. A regressão logística terá `max_iter=1000` para permitir que o algoritmo convirja. A floresta começa com 100 árvores (valor padrão), `random_state=42` para reprodutibilidade e pesos balanceados.

In [3]:
model_specs = {
    "dummy": DummyClassifier(strategy="most_frequent", random_state=42),
    "logistic_regression": LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        random_state=42,
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    ),
}

for name, estimator in model_specs.items():
    print(f"{name}: {estimator}")

dummy: DummyClassifier(random_state=42, strategy='most_frequent')
logistic_regression: LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
random_forest: RandomForestClassifier(class_weight='balanced', n_jobs=-1, random_state=42)


## Ajustar e salvar cada pipeline

Cada estimador recebe seu próprio `ColumnTransformer`. O `.fit()` é chamado apenas com `X_train` e `y_train`; assim, medianas, escalas, categorias e parâmetros do classificador são aprendidos a partir do treino. Guardamos o pré-processamento e o classificador juntos em um `Pipeline`, para que o artefato salvo aplique sempre as mesmas transformações quando for carregado.

`joblib` serializa os objetos Scikit-learn. Os arquivos são salvos com compressão moderada em `results/models/`, para reduzir espaço sem alterar o conteúdo do pipeline. Ainda não calculamos métricas nem usamos o teste para previsões.

In [4]:
fitted_models = {}
model_paths = {}

for name, estimator in model_specs.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", make_preprocessor(X_train)),
            ("classifier", estimator),
        ]
    )
    pipeline.fit(X_train, y_train)

    model_path = MODELS_PATH / f"{name}_pipeline.joblib"
    joblib.dump(pipeline, model_path, compress=3)
    fitted_models[name] = pipeline
    model_paths[name] = model_path

    classifier = pipeline.named_steps["classifier"]
    print(
        f"{name}: treinado em {len(X_train):,} linhas; "
        f"classes={classifier.classes_.tolist()}; salvo em {model_path.relative_to(PROJECT_ROOT)} "
        f"({model_path.stat().st_size:,} bytes)"
    )

dummy: treinado em 32,950 linhas; classes=['no', 'yes']; salvo em results\models\dummy_pipeline.joblib (2,461 bytes)


logistic_regression: treinado em 32,950 linhas; classes=['no', 'yes']; salvo em results\models\logistic_regression_pipeline.joblib (3,172 bytes)


random_forest: treinado em 32,950 linhas; classes=['no', 'yes']; salvo em results\models\random_forest_pipeline.joblib (16,372,239 bytes)


## Conferir que os pipelines podem ser carregados

Carregamos os artefatos salvos para confirmar que cada arquivo contém um pipeline completo. Isso não avalia desempenho; a comparação de métricas ocorrerá no notebook 04, usando o teste na avaliação final.

In [5]:
loaded_models = {
    name: joblib.load(path)
    for name, path in model_paths.items()
}

for name, pipeline in loaded_models.items():
    print(
        f"{name}: Pipeline com etapas "
        f"{list(pipeline.named_steps)}; classificador={type(pipeline.named_steps['classifier']).__name__}"
    )

dummy: Pipeline com etapas ['preprocessor', 'classifier']; classificador=DummyClassifier
logistic_regression: Pipeline com etapas ['preprocessor', 'classifier']; classificador=LogisticRegression
random_forest: Pipeline com etapas ['preprocessor', 'classifier']; classificador=RandomForestClassifier


## Síntese

- Três modelos foram treinados somente no conjunto de treino.
- O Dummy estabelece uma referência majoritária.
- Os dois modelos principais usam pesos balanceados, uma decisão inicial motivada pelo desbalanceamento que será avaliada com precision, recall, F1, ROC-AUC e curvas de precisão-recall.
- Os pipelines completos foram salvos com `joblib` em `results/models/`.
- Nenhum modelo foi escolhido como vencedor nesta etapa. O teste continua reservado para a avaliação final.
- Não houve busca de hiperparâmetros nem ajuste de limiar. Se forem feitos ajustes, a validação cruzada (`StratifiedKFold`, `random_state=42`) ocorrerá somente no conjunto de treino. Isso evita adaptar o modelo ao teste e produzir uma estimativa excessivamente otimista.